# Big Data Platforms — Lecture 4 Notes

Course: DATA140031 (MOOC, 3 ECTS) + DATA140032 (MOOC Exam, 2 ECTS)
Lecturer: Keijo Heljanko, Department of Computer Science, University of Helsinki
Date: 10.9.2026

Lecture 3 ended on HDFS replicating everything three times and asked, more or
less, "why not just use RAID instead?" Lecture 4 is the answer to that question.
It works through RAID levels one by one, hits the actual failure mode that makes
plain RAID 5/6 risky (the write hole), and then connects that all the way back to
why HDFS is built around write-once storage and three-way replication in the
first place. By the end it's clear HDFS isn't reinventing RAID for no reason —
it's RAID 10-ish redundancy plus a database-style transaction log, applied at the
scale of a whole cluster instead of a single server's disk shelf.


## 1. Why storage needs fault tolerance at all

Hard disks fail — that's the starting assumption, not an edge case. Field studies
report around a **3% average yearly replacement rate** for hard disks in
large-scale systems. That number sounds small until you multiply it by a cluster
with a few thousand disks: at 3% a year, a 3,000-disk cluster is looking at
roughly 90 disk failures annually — close to two failures a week, on average,
forever. Any storage design at that scale has to treat disk failure as a routine
event, not an emergency.

The lecture lists three ways of building reliable storage on top of unreliable
disks, roughly in order of increasing scale:

1. **RAID** — redundancy inside a single server or storage array.
2. **Replication or erasure coding inside a datacenter** — HDFS's three-way
   default replication, spread across at least two racks, is the running example.
3. **Geographic replication across datacenters** — Amazon S3's cross-region
   replication is the example given here.

The rest of the lecture mostly lives at level 1 (RAID) before connecting it back
up to level 2 (HDFS).


## 2. RAID — Redundant Array of Independent Disks

RAID is still the most common fault-tolerance mechanism at small scale — inside a
single server, or in dedicated SAN/NAS hardware. The "I" originally stood for
*inexpensive*, back when the whole pitch was that a pile of cheap disks could
match one expensive one; these days it's usually read as *independent*.

**RAID 0 (striping)** isn't actually a fault-tolerance mechanism at all — it's a
performance trick that gets lumped in with the RAID family anyway. Data is split
into stripes and spread across many disks so reads and writes can happen in
parallel. That's the entire point: more disks working simultaneously means more
throughput. It offers **zero** redundancy — lose one disk in the array and every
file that had a stripe on it is gone, because each stripe was only ever stored
once. This makes RAID 0 a genuinely bad choice for anything you'd be upset to
lose, and a genuinely good choice for scratch space or data you can regenerate
cheaply (temp files, caches, intermediate build artifacts).


![RAID 0 striping](images/raid0.png)


The remaining levels below are the ones actually used **for** fault
tolerance:

- **RAID 1** — mirroring
- **RAID 5** — block-level striping with distributed parity
- **RAID 6** — block-level striping with double distributed parity
- **RAID 10** — a nested level, a stripe of mirrored pairs

Further reading, and the source of the RAID diagrams used in this lecture:
<http://en.wikipedia.org/wiki/Standard_RAID_levels>

One piece of terminology worth pinning down before going further: **IOPS**
(I/O operations per second) — the number of small, random reads or writes a
system can do per second. This is a different number from raw sequential
throughput, and it's the number that ends up mattering most for the RAID 5/RAID 6
write penalty discussed below.


## 3. RAID 1 — Mirroring

Every block gets written to **two** disks: a master copy first, then a mirror
(slave) copy written right after. Reads can be served from either disk
interchangeably, which is a nice side benefit — you effectively get to load-balance
reads across two drives.

The cost is straightforward: you lose half your raw storage capacity, and writes
cost double the bandwidth/IOPS of a single drive, since every write has to happen
twice.


![RAID 1 mirroring](images/raid1.png)


**Data availability:** as long as either disk survives, the data survives.

**Repair is simple, at least conceptually:** swap in a replacement drive and copy
all the data across from the surviving mirror. No parity math involved anywhere —
just a straight copy.


## 4. RAID 5 — block-level striping with distributed parity

Data lives across *n + 1* disks. For every *n* blocks of real data, one parity
checksum block is stored, and — this is the "distributed" part — the parity
blocks themselves rotate across all the disks rather than all sitting on one
dedicated parity drive. RAID 5 tolerates exactly **one** disk failure at a time.


![RAID 5 block-level striping with distributed parity](images/raid5.png)


### RAID 5 in practice

- Sequential reads and writes stripe naturally across all disks, so sequential
  throughput and random *read* IOPS are both good.
- Storage overhead is minimal — only one disk's worth of capacity goes to parity,
  regardless of how many disks are in the array.
- **Small random writes are the weak point.** A single small write that touches
  one data block has to: read the old data block, read the old parity block,
  write the new data block, write the new parity block. That's **4× the IOPS**
  of a plain write, in the worst case. Battery-backed caches exist specifically to
  soften this overhead by buffering and batching those operations.
- **Rebuilds are slow and get slower with more disks.** Reconstructing a failed
  disk means reading *every block of every other disk* in the array to recompute
  the missing data via parity. And while that rebuild is in progress, every read
  that would have hit the missing disk instead has to read all *n* surviving
  disks and recompute the value on the fly — so performance craters exactly when
  you can least afford it (a second failure during that window means real data
  loss). This vulnerability — plus how long rebuilds take on today's large-capacity
  drives — is why most vendors now steer people toward RAID 6 instead, especially
  when using large-capacity SATA drives where rebuild windows stretch out for
  hours or days.

### The RAID 5 write hole

This is the failure mode worth understanding properly, because it comes back
later when explaining why HDFS is built the way it is.

A single logical write to a RAID 5 stripe actually has to touch *multiple*
physical disks — the data block(s) plus the parity block — and there's no way to
make all of those individual disk writes happen at the exact same instant. If
power fails partway through, some of those writes land and others don't. The
array is now left with data blocks and a parity block that no longer agree with
each other, and nothing about the RAID controller can tell after the fact which
writes actually completed.

The nasty consequence isn't just "some data on that stripe might be off" — it's
that if a *different* disk then fails, the recovery process uses that now-wrong
parity to try to regenerate the missing disk's contents. Garbage parity produces
garbage reconstructed data. The corruption spreads to data that was never
directly involved in the original interrupted write.

Battery-backed caches and UPS units exist largely to close this gap — either by
making sure the cache survives a power loss long enough to finish the write later,
or by preventing the power loss from happening in the first place. Separately,
RAID 5 also doesn't protect against a disk quietly returning bad data during a
parity calculation — this is exactly what "hot spares" are for, to shrink the
window during which a bad block can cause real damage.


## 5. RAID 6 — block-level striping with double distributed parity

Same idea as RAID 5, but with **two** independent parity blocks per stripe
instead of one, stored across *n + 2* disks. This buys tolerance for **two**
simultaneous disk failures instead of one.


![RAID 6 block-level striping with double distributed parity](images/raid6.png)


### RAID 6 in practice

- Same good sequential and random-read behaviour as RAID 5.
- Storage overhead: two disks' worth given up to parity instead of one.
- **The write penalty is worse than RAID 5, not just doubled naively** — a small
  random write needs to read the data block plus *both* parity blocks, then write
  back all three modified blocks: **6× the IOPS** in the worst case.
- **Rebuild is slower still**, for the same fundamental reason as RAID 5 — every
  missing block read during a rebuild requires reading all *n* surviving data
  disks — but RAID 6 can absorb *one additional* disk failure during that rebuild
  window and still recover, since the surviving *n* disks are enough to
  reconstruct any two missing ones.
- **RAID 6 has its own write hole**, structurally identical to RAID 5's: a power
  failure mid-write to a stripe that spans *n + 2* disks can leave things
  inconsistent, and the same battery-backed-cache/UPS mitigations apply. It also
  shares RAID 5's blind spot around a single disk quietly returning corrupted data
  during parity math.


## 6. RAID 10 — a stripe of mirrors

RAID 10 is a *nested* RAID level — RAID 0 (striping) built on top of RAID 1
(mirroring) pairs, rather than a scheme in its own right. Data lives on *2n*
disks: each mirrored pair holds an identical copy of its share of the data, and
the pairs are then striped together into one logical volume.

Reference on nested levels generally: <http://en.wikipedia.org/wiki/Nested_RAID_levels>


![RAID 10 — stripe of mirrors](images/raid10.png)


### RAID 10 in practice

- Worst case, it tolerates exactly **one** disk failure — but the best case is
  much better: it can survive losing *one disk from every single mirror pair*
  simultaneously (that's *n* failures out of *2n* disks), as long as you never
  lose *both* disks of the same pair. Whether you land in the best or worst case
  is down to luck, not design.
- Loses half of total storage capacity, same trade-off as plain mirroring.
- Sequential and random performance are both good — reads and writes stripe
  across all the underlying disks, and because there's no parity math at all,
  small random writes only cost **2× IOPS** (write to both sides of the mirror),
  a fraction of RAID 5's 4× or RAID 6's 6×.
- **Rebuild is comparatively quick**: reconstructing a failed disk just means
  copying data straight over from its surviving mirror partner — no need to read
  and recompute from every other disk in the array the way parity-based RAID
  requires. Hot spares are still worth having, to start that copy as early as
  possible.
- It **cannot** survive losing both disks in the same mirror pair — that data is
  simply gone.
- **The write hole is avoidable here, conditionally.** If writes to the master
  drive and its mirror slave are strictly ordered (master first, then slave), an
  unclean shutdown just means replaying the last few known-good writes from
  master to slave — no ambiguity about what the correct state should be. Without
  that ordering guarantee, there's no way to tell which of the two mirror disks
  holds the more recent version after an unclean shutdown, so a UPS is still
  strongly recommended even on RAID 10.
- Like every RAID level discussed here, RAID 10 still does nothing to stop a
  single drive quietly returning corrupted data during rebuild.


## 7. Choosing a RAID level, and the obligatory reminder

| Level | Use case |
|---|---|
| RAID 0 | Temporary/scratch space for data you can regenerate |
| RAID 1 | Small installations, server boot disks |
| RAID 5 | Some fault tolerance, minimal storage overhead; weak on small random writes; needs the write-hole mitigations |
| RAID 6 | Fairly strong fault tolerance, reasonable overhead; small random writes hurt even more than RAID 5; same write-hole concerns |
| RAID 10 | Weaker fault tolerance than RAID 6 but stronger than RAID 5; gives up half the capacity; excellent small-random-write performance, which is exactly why it's the usual pick for database workloads; can dodge the write hole entirely under strict write ordering |

The line worth repeating to anyone who's ever mixed up "redundant" with "backed
up": **RAID is no substitute for backups.** RAID protects against a *disk*
failing. It does nothing whatsoever against someone deleting the wrong file,
ransomware encrypting everything in place, or a bug that silently corrupts data
before it's ever written to disk — RAID will happily replicate corrupted data
just as faithfully as good data.

On hardware requirements: RAID 1 and RAID 10 can be made safe purely through
software configuration (correct write ordering), without needing specialised
hardware — though a UPS is still recommended regardless. RAID 5 and RAID 6 are
more storage-efficient than RAID 10 for the same fault tolerance, but they
genuinely need specialised hardware (a battery-backed cache) to be safe against
corruption from an unclean shutdown, precisely because of the write hole.


## 8. The RAID write hole, properly explained

Having named the problem for RAID 5, 6, and 10 individually, it's worth stepping
back and asking: what's actually going on here, structurally?

RAID 5 and RAID 6 both need to update **all** the drives in a stripe atomically —
either every write in the stripe update lands, or none of them do — in order to
stay internally consistent. That's exactly the same requirement databases have
always had, and databases solved it decades ago with **atomic transactions**.

How do databases actually implement that guarantee? By keeping a **persistent
log** of the modifications about to be performed, so that if the system dies
mid-update, it can replay whatever modifications didn't make it through once it
comes back up. RAID 5/6 need precisely that same mechanism — a persistent
transaction log — in order to offer atomic stripe updates.

Two concrete ways this gets implemented in practice:

1. **A battery-backed cache** in the RAID controller — the hardware approach.
2. **A log-structured, storage-aware filesystem** — the software approach. The
   two well-known examples are Sun/Oracle's **ZFS** and Linux's **Btrfs**. Both
   are given full control over the raw disk devices and use a **copy-on-write**
   transactional object model: instead of ever modifying a block in place, they
   write the new version somewhere fresh and only atomically flip a pointer once
   the new version is fully committed. If the power dies mid-write, the old,
   still-consistent version is simply still there — there's no half-updated
   stripe to worry about, because nothing was ever updated in place to begin
   with. ZFS additionally checksums everything, which is a separate defence
   against the "disk silently returns bad data" problem mentioned earlier.

There's a memorable one-liner behind all of this, from database pioneer **Jim
Gray**: *"Update in place is a poison apple."* (Gray, *The Transaction Concept:
Virtues and Limitations*, invited paper, VLDB 1981, pp. 144–154.) The bottom
line the lecture draws from it: reliable *distributed* updates fundamentally
require database-style transaction logging to make stripe updates atomic — there
isn't a way around this that doesn't eventually reduce to that same idea in
disguise.


In [1]:
# A tiny illustration of the copy-on-write idea behind ZFS/Btrfs and, by
# extension, why HDFS's write-once model sidesteps the write hole entirely.
# This is a toy simulation, not real filesystem code -- just enough to make
# the "never update in place" idea concrete.

class CowBlock:
    """A copy-on-write block: 'modifying' it never touches the old version."""
    def __init__(self, data):
        self.data = data

    def write(self, new_data):
        # Instead of mutating self.data in place, we return a brand new block.
        # The caller is responsible for atomically repointing to it.
        return CowBlock(new_data)


# "Filesystem" state: one pointer to the current committed block.
current = CowBlock("v1: original stripe contents")

# Simulate an update that gets interrupted by a power loss halfway through.
staged = current.write("v2: updated stripe contents")

power_loss_before_commit = True

if power_loss_before_commit:
    # The pointer was never flipped, so 'current' still points at a fully
    # consistent v1. No write hole -- nothing was ever half-updated in place.
    print("After crash, filesystem sees:", current.data)
else:
    current = staged
    print("After crash, filesystem sees:", current.data)


After crash, filesystem sees: v1: original stripe contents


## 9. Scaling up vs. scaling out storage

Once you need more capacity than one server's worth of disks, there are two
directions to go, and each brings its own problems.

**Scaling up: buy a bigger box.** Large NAS/SAN systems can handle thousands of
disks behind specialised, purpose-built hardware.

- These are not commodity hardware — pricing is very high relative to raw
  storage capacity.
- Performance often lags behind a well-designed scale-out approach, ironically,
  given the price tag.
- Still often the right call for genuinely modest storage needs, where the
  operational simplicity of "one box" outweighs the cost premium.

**Scaling out: use many small servers instead.** This introduces a different set
of problems.

- Ordinary hardware RAID inside each server handles individual *disk* failures
  fine, but does nothing to protect against a whole *server* or *RAID
  controller* dying.
- With hundreds of servers and RAID controllers in the fleet, those failures stop
  being rare edge cases and become routine — the same logic as the 3% disk
  failure rate, just one level up the stack.
- To keep data available while a failed server or controller gets repaired, the
  data has to be replicated across *multiple* servers, not just protected within
  one.

Which leads to the punchline motivating the rest of this lecture: **if data has
to be replicated across servers anyway, why also pay for expensive hardware RAID
controllers inside each one?** Just build the redundancy into software instead —
a "distributed fault-tolerant storage" layer. HDFS is the prime example of
exactly that.


## 10. HDFS design decisions, read through a RAID lens

This is the payoff of going through RAID in this much detail — HDFS's storage
layout is, structurally, a RAID level applied across whole machines instead of
across disks in one box.

**HDFS replication ≈ RAID 10, plus checksums.** Each HDFS block is simply
replicated onto multiple *DataNodes* — full copies, no parity math anywhere,
exactly the way RAID 1/10 mirroring works. On top of that, HDFS adds
sub-block **CRC-32 checksums** for data integrity, catching the "disk silently
returns bad data" failure mode that plain RAID mirroring doesn't address on its
own.

- For **small installations**, replicating each block **twice** is enough — and
  that's *literally* a RAID 10 layout, just spread across two machines instead of
  two disks in the same box.
- For **large installations**, HDFS's usual default of replicating each block
  **three** times lines up with configuring Linux software RAID 10 with three
  disks per mirror set instead of two — which tolerates two disk failures per
  mirror set instead of one. Same underlying idea, just with the redundancy
  factor turned up because at larger scale, the odds of a second concurrent
  failure go up too.

**So why not use RAID 6 for HDFS instead, and save on storage overhead?** RAID 6
only gives up two disks' worth of capacity to parity no matter how wide the
stripe is, versus HDFS's flat 3× overhead from full replication — which on paper
looks like a large efficiency win.

**This is exactly where erasure coding enters the picture.**


## 11. Erasure codes

Many RAID 6 implementations already use an **erasure code** under the hood — most
commonly a **Reed-Solomon code** — to compute parity and to recover from up to
two missing disks. What makes erasure codes more interesting than "just RAID 6
under a fancier name" is that they generalise cleanly: an erasure code can be
built to recover from *any* number of missing pieces, not just two, simply by
adding more parity blocks. This general category is called **forward error
correction**, and it shows up in plenty of unrelated places too — Reed-Solomon
codes, for instance, are the same family used for error correction on audio CDs.

Further reading: <http://en.wikipedia.org/wiki/Erasure_code> and
<http://en.wikipedia.org/wiki/Reed%E2%80%93Solomon_error_correction>

### Erasure codes at production scale

Windows Azure Storage runs erasure coding in production — see Cheng Huang et
al., *Erasure Coding in Windows Azure Storage*, USENIX ATC'12
(<https://www.usenix.org/system/files/conference/atc12/atc12-final181_0.pdf>).
One concrete scheme from that world: **10 data blocks + 4 parity blocks**, a
**1.4× storage overhead**, tolerating **four** simultaneous disk failures. Compare
that to HDFS's default 3× overhead for tolerating two failures (since the third
copy is what lets you survive losing two out of three) — the erasure-coded scheme
gets you *more* fault tolerance for *less* than half the storage cost.

### HDFS-EC

HDFS itself picked this up as **HDFS-EC**, released in HDFS 3.1:
<https://hadoop.apache.org/docs/stable/hadoop-project-dist/hadoop-hdfs/HDFSErasureCoding.html>

It's essentially RAID 6 taken further — more parity blocks, tolerating more
simultaneous disk failures — and it inherits exactly the same trade-off that
governs the choice between RAID 10 and RAID 6 in the first place: rebuild
efficiency (full-copy replication wins) versus storage efficiency (erasure coding
wins). HDFS-EC doesn't force a cluster-wide choice between the two, either — the
replication vs. erasure-coding policy can be set independently **per directory**,
so a cluster can keep hot, frequently-rewritten data on plain 3× replication for
fast rebuilds while parking cold, rarely-touched data under erasure coding to
save space.


## 12. How HDFS actually avoids the write hole

Given everything above about RAID 5/6 needing a persistent transaction log to
stay safe, the obvious question is: does HDFS need one too?

**The short answer is no — because HDFS sidesteps the problem entirely by
disallowing the thing that causes it.** The core move is avoiding "update in
place" altogether, via the **write-once-read-many (WORM)** pattern already
covered in lecture 3: once a data block is written, it's never modified again.

Recall the general principle: the only real way to avoid the RAID write hole is
to bring genuine database-style transactions into the picture, whether that
lives in a hardware RAID controller's battery-backed cache or in a
copy-on-write filesystem like ZFS or Btrfs that never touches old data in place.
HDFS's centralised **NameNode** gives it a natural place to do exactly that: it
can log filesystem-state transactions the same way a database logs its
transactions, because there's a single, well-defined place all metadata changes
flow through.

But HDFS goes one step further than ZFS/Btrfs in cutting down how much
transaction logging is actually needed: **because stripe (block) updates are
disallowed outright, no transaction is ever needed for one.** There's nothing to
make atomic, because the operation that would need atomicity — modifying an
existing block in place — simply never happens. HDFS only needs to log
transactions around file **close** time, when a file's set of blocks becomes
final. Compare that to RAID 5/6, which have to guard against a write hole on
*every single stripe write*, precisely because in-place updates are the whole
point of a general-purpose block device. HDFS doesn't have that problem because
it never promised to be a general-purpose block device in the first place.


## 13. Hard disk read errors and why checksums matter at scale

One more failure mode, orthogonal to whole-disk failure: **unrecoverable read
errors (UREs)**. When a URE happens, the disk isn't dead — it just flatly can't
read back one particular block, and says so.

- Consumer-grade disks are typically specified for URE rates of at most **1 error
  per 10¹⁵ bits read**.
- Some enterprise-grade disks push that out to **1 error per 10¹⁶ bits read** —
  ten times better, but still not zero.

Those numbers sound comfortably tiny until you multiply by the scale a large
storage system operates at. **10¹⁵ bits is about 125 terabytes** — and a large
storage cluster reads *far* more than that over its lifetime, often that much in
a single day. At that scale, unrecoverable read errors aren't a hypothetical edge
case anymore; they're something that will happen, regularly, as a simple
consequence of how much data flows through the system.

Which is exactly why extra checksumming and error correction, layered on top of
whatever the disk itself provides, become necessary once you're operating at
scale:

- **HDFS** stores a **CRC-32** checksum for every **512 bytes** of data — that
  512-byte figure isn't arbitrary, it's the smallest unit HDFS will ever do a
  random read against, so it's the finest granularity at which corruption
  actually needs to be detectable.
- **ZFS** checksums the *entire* filesystem, not just fixed sub-blocks. **Btrfs**
  does the same on Linux.

For further reading on RAID 6 and where it's heading, the lecture points to:
Adam Leventhal, *Triple-Parity RAID and Beyond*, ACM Queue 7(11): 30 (2009).


In [2]:
# A quick sanity check on the 10^15-bits number quoted above, just to see
# how close "one URE" is to something you'd actually hit at scale.

bits_per_byte = 8
bits_per_terabyte = 10**12 * bits_per_byte  # decimal TB, as drive vendors use it

consumer_ure_rate_bits = 10**15
enterprise_ure_rate_bits = 10**16

print(f"Consumer URE threshold:   {consumer_ure_rate_bits / bits_per_terabyte:.1f} TB read")
print(f"Enterprise URE threshold: {enterprise_ure_rate_bits / bits_per_terabyte:.1f} TB read")

# A modest HDFS cluster doing, say, 50 TB of reads a day would then expect to
# cross the *consumer* URE threshold roughly every ~2.5 days -- illustrating
# why per-block checksums aren't optional at that scale, they're load-bearing.
daily_reads_tb = 50
days_to_consumer_ure = (consumer_ure_rate_bits / bits_per_terabyte) / daily_reads_tb
print(f"\nAt {daily_reads_tb} TB/day of reads, expect a consumer-grade URE roughly every "
      f"{days_to_consumer_ure:.1f} days")


Consumer URE threshold:   125.0 TB read
Enterprise URE threshold: 1250.0 TB read

At 50 TB/day of reads, expect a consumer-grade URE roughly every 2.5 days


## Summary

Lecture 3 established that HDFS replicates every block three times and never
updates a block in place. Lecture 4 is really the justification for both of those
choices, worked backward from first principles: RAID 5 and RAID 6 are more
storage-efficient than mirroring, but they carry a structural weakness — the
write hole — that only goes away with database-style transaction logging, which
in turn only exists in practice as either specialised battery-backed hardware or
a copy-on-write filesystem like ZFS/Btrfs. HDFS takes the more radical option:
rather than solving the write-hole problem, it makes the problem impossible by
banning in-place updates altogether, and it gets to lean on a centralised
NameNode as a natural place to log the transactions it *does* still need — which
turns out to be far fewer than a general-purpose filesystem would require, since
there's nothing to make atomic once stripe updates simply aren't allowed. Add
per-block checksums on top (because at real scale, unrecoverable read errors stop
being a rounding error), and erasure coding as an option where the extra storage
overhead of full replication isn't worth paying, and you land pretty much exactly
on the system described in lecture 3 — just now with the reasoning attached
instead of taken as given.
